# Generación del dataset de flujos Mirai

Construye `flows.csv` a partir de las capturas PCAP en bruto. Solo hace falta ejecutarlo una vez.

In [2]:
# configuración: imports y carga de archivos
import os
import pandas as pd
import subprocess
from pathlib import Path
from collections import defaultdict
import warnings; warnings.filterwarnings('ignore')
import dpkt

EDITCAP   = r'C:\Program Files\Wireshark\editcap.exe'
DATA_PATH = str(Path.home() / ".cache" / "kagglehub" / "datasets" / "nguyenquanitmo" / "mirai-raw-dataset" / "versions" / "2")
TEMP_DIR  = str(Path.cwd().parent / "data" / "temp_pcap")
OUT_CSV   = str(Path.cwd().parent / "data" / "flows.csv")

os.makedirs(TEMP_DIR, exist_ok=True)

FILE_REGISTRY = []

# non-attack = tráfico Mirai C&C (escaneo/infección/control) → MALICIOSO (label=1)
for i in range(1, 31):
    FILE_REGISTRY.append((f'non-attack{i}.pcapng', 'Mirai_CnC', 1))

# non-botnet = tráfico de cámara IP limpia → NORMAL (label=0)
for i in range(1, 31):
    if i == 16:
        continue
    FILE_REGISTRY.append((f'non-botnet{i}.pcapng', 'Normal', 0))

ATTACK_FILES = [
    ('ack_attack',   'ACK_Flood',   1),
    ('dns_attack',   'DNS_Flood',   1),
    ('greip_attack', 'GREIP_Flood', 1),
    ('http_attack',  'HTTP_Flood',  1),
    ('syn_attack',   'SYN_Flood',   1),
    ('udp_attack',   'UDP_Flood',   1),
    ('vse_attack',   'VSE_Flood',   1),
]
for base, class_name, label in ATTACK_FILES:
    for i in range(1, 4):
        FILE_REGISTRY.append((f'{base}{i}.pcapng', class_name, label))

PORT_TO_STATE = {53: 'DNS', 80: 'HTTP', 443: 'HTTPS', 22: 'SSH'}

def get_state(proto, dport):
    if proto == 6:
        return PORT_TO_STATE.get(dport, 'TCP_OTHER')
    elif proto == 17:
        return PORT_TO_STATE.get(dport, 'UDP_OTHER')
    else:
        return 'OTHER'

def convert_to_pcap(pcapng_path):
    basename  = os.path.splitext(os.path.basename(pcapng_path))[0]
    pcap_path = os.path.join(TEMP_DIR, f'{basename}.pcap')
    subprocess.run(
        [EDITCAP, '-F', 'pcap', pcapng_path, pcap_path],
        capture_output=True, timeout=120, check=True
    )
    return pcap_path

def extract_flows(pcap_path, class_name, label):
    flows = defaultdict(lambda: {
        'timestamps': [], 'states': [],
        'n_pkts': 0, 'n_bytes': 0,
        'f_pkts': 0, 'f_bytes': 0,
        'b_pkts': 0, 'b_bytes': 0,
    })
    with open(pcap_path, 'rb') as f:
        reader = dpkt.pcap.Reader(f)
        for ts, raw in reader:
            try:
                eth = dpkt.ethernet.Ethernet(raw)
                if not isinstance(eth.data, dpkt.ip.IP):
                    continue
                ip      = eth.data
                proto   = ip.p
                pkt_len = len(raw)
                if isinstance(ip.data, (dpkt.tcp.TCP, dpkt.udp.UDP)):
                    sport = ip.data.sport
                    dport = ip.data.dport
                else:
                    sport = dport = 0
                fwd_key = (ip.src, ip.dst, sport, dport, proto)
                bwd_key = (ip.dst, ip.src, dport, sport, proto)
                if bwd_key in flows:
                    flows[bwd_key]['b_pkts']  += 1
                    flows[bwd_key]['b_bytes'] += pkt_len
                    flows[bwd_key]['n_pkts']  += 1
                    flows[bwd_key]['n_bytes'] += pkt_len
                else:
                    key = fwd_key
                    flows[key]['timestamps'].append(ts)
                    flows[key]['states'].append(get_state(proto, dport))
                    flows[key]['n_pkts']  += 1
                    flows[key]['n_bytes'] += pkt_len
                    flows[key]['f_pkts']  += 1
                    flows[key]['f_bytes'] += pkt_len
            except Exception:
                continue
    records = []
    for (src_ip, dst_ip, sport, dport, proto), d in flows.items():
        if not d['timestamps']:
            continue
        ts_arr   = d['timestamps']
        duration = ts_arr[-1] - ts_arr[0] if len(ts_arr) > 1 else 0.0
        records.append({
            'src_ip':       src_ip,
            'flow_start':   ts_arr[0],
            'duration':     duration,
            'state':        d['states'][0],
            'n_pkts':       d['n_pkts'],
            'n_bytes':      d['n_bytes'],
            'f_pkts':       d['f_pkts'],
            'f_bytes':      d['f_bytes'],
            'b_pkts':       d['b_pkts'],
            'b_bytes':      d['b_bytes'],
            'avg_pkt_size': d['n_bytes'] / d['n_pkts'],
            'class_name':   class_name,
            'label':        label,
        })
    return records

# Guard: no regenerar si flows.csv ya existe (pon REGENERAR=True para forzar)
REGENERAR = False
if (not REGENERAR) and Path(OUT_CSV).exists():
    print(f'{OUT_CSV} ya existe; no se regenera (REGENERAR=False).')
    df = pd.read_csv(OUT_CSV)
    print(f'Shape: {df.shape}')
else:
    all_records = []
    for i, (filename, class_name, label) in enumerate(FILE_REGISTRY):
        pcapng_path = os.path.join(DATA_PATH, filename)
        print(f'[{i+1}/{len(FILE_REGISTRY)}] {filename}  ({class_name})')
        try:
            pcap_path = convert_to_pcap(pcapng_path)
            records   = extract_flows(pcap_path, class_name, label)
            os.remove(pcap_path)
            all_records.extend(records)
            print(f'    {len(records):,} flujos')
        except Exception as e:
            print(f'    ERROR: {e}')

    df = pd.DataFrame(all_records)
    df.to_csv(OUT_CSV, index=False)
    print(f'\nDataset  : {OUT_CSV}')
    print(f'Shape    : {df.shape}')
    print(f'Anomalías: {df["label"].mean():.1%}')
    print(df['class_name'].value_counts().to_string())


C:\Users\user\markov_mirai\flows.csv ya existe; no se regenera (REGENERAR=False).
Shape: (1006776, 13)
